In [ ]:
import os
from tqdm import tqdm
import json
import re
from pathlib import Path
import pandas as pd

In [ ]:
def generate_annotated_tokens(input_tokens, entities_df, opening_ent="<ent id=COREF_", closing_ent="</ent>", opening_zero_ent="</zero_ent id=COREF_"):

    annotated_text_dict = {i: word for i, word in enumerate(input_tokens)}

    zero_entities_df = entities_df[entities_df["cat"] == "zero_node"].copy()
    zero_entities_df = zero_entities_df.sort_values(by=["start_token", "COREF"], ascending=[True, True]).reset_index(drop=True)  # constructed from inside out
    for start_token, cat, COREF in zero_entities_df[['start_token', 'cat', 'COREF']].values:
        zero_tag = f"{opening_zero_ent}{COREF}>"
        annotated_text_dict[start_token] = f"{annotated_text_dict[start_token]} {zero_tag}"

    non_zero_entities_df = entities_df[entities_df["cat"] == "entity"].copy()
    non_zero_entities_df = non_zero_entities_df.sort_values(by=["start_token", "mention_len", "COREF"],
                                          ascending=[True, True, False]).reset_index(drop=True)  # constructed from inside out
    for start_token, cat, COREF in non_zero_entities_df[['start_token', 'cat', 'COREF']].values:
        opening_tag = f"{opening_ent}{COREF}>"
        annotated_text_dict[start_token] = f"{opening_tag} {annotated_text_dict[start_token]}"

    non_zero_entities_df = non_zero_entities_df.sort_values(by=["end_token", "mention_len", "COREF"],
                                          ascending=[True, True, True]).reset_index(drop=True)
    for end_token, cat, COREF in non_zero_entities_df[['end_token', 'cat', 'COREF']].values:
        closing_tag = closing_ent
        annotated_text_dict[end_token] = f"{annotated_text_dict[end_token]} {closing_tag}"

    annotated_tokens = list(annotated_text_dict.values())
    return annotated_tokens

In [ ]:
def CRAC_plaintext_annotation(input_tokens, entities_df):
    entities_df["COREF"] = entities_df["COREF"].apply(lambda x: f'e{x+1}')

    annotated_text_dict = {i: word for i, word in enumerate(input_tokens)}

    non_zero_entities = entities_df[(entities_df["cat"] == "entity") & (entities_df["mention_len"] != 1)].copy()

    for start_token, cat, COREF in non_zero_entities[['start_token', 'cat', 'COREF']].values:
        tag = f"|[{COREF}"
        annotated_text_dict[start_token] = f"{annotated_text_dict[start_token]}{tag}"
        annotated_text_dict[start_token] = re.sub(r'(\[e\d+)\|\[e', r'\1,[e', annotated_text_dict[start_token])
        # .replace("]|", "],")

    single_word_non_zero_entities = entities_df[(entities_df["cat"] == "entity") & (entities_df["mention_len"] == 1)].copy()
    for start_token, cat, COREF in single_word_non_zero_entities[['start_token', 'cat', 'COREF']].values:
        tag = f"|[{COREF}]"
        annotated_text_dict[start_token] = f"{annotated_text_dict[start_token]}{tag}"
        annotated_text_dict[start_token] = re.sub(r'(\[e\d+)\|\[(e\d+)', r'\1,[\2', annotated_text_dict[start_token])

    non_zero_entities = non_zero_entities.sort_values(by=["start_token", "mention_len", "COREF"],
                                      ascending=[True, True, True]).reset_index(drop=True)
    for end_token, cat, COREF in non_zero_entities[['end_token', 'cat', 'COREF']].values:
        tag = f"|{COREF}]"
        annotated_text_dict[end_token] = f"{annotated_text_dict[end_token]}{tag}"
        annotated_text_dict[end_token] = re.sub(r'(e\d+)\]\|(e\d+)\]', r'\1],\2]', annotated_text_dict[end_token])

    zero_entities = entities_df[(entities_df["cat"] == "zero_node")].copy()
    for start_token, cat, COREF in zero_entities[['start_token', 'cat', 'COREF']].values:
        tag = f" ##|[{COREF}]"
        annotated_text_dict[start_token] = f"{annotated_text_dict[start_token]}{tag}"

    annotated_tokens = list(annotated_text_dict.values())

    # annotated_sentence = " ".join(annotated_tokens)
    return annotated_tokens

In [ ]:
def generate_head_only_annotations(input_tokens, entities_df, entity_tag_prefix="<ent", zero_tag_prefix="<zero"):

    annotated_text_dict = {i: word for i, word in enumerate(input_tokens)}

    zero_entities_df = entities_df[entities_df["cat"] == "zero_node"].copy()
    zero_entities_df = zero_entities_df.sort_values(by=["start_token", "COREF"], ascending=[True, True]).reset_index(drop=True)  # constructed from inside out
    for start_token, cat, COREF in zero_entities_df[['start_token', 'cat', 'COREF']].values:
        zero_tag = f"{zero_tag_prefix}{COREF}>"
        annotated_text_dict[start_token] = f"{annotated_text_dict[start_token]} {zero_tag}"

    non_zero_entities_df = entities_df[entities_df["cat"] == "entity"].copy()
    non_zero_entities_df = non_zero_entities_df.sort_values(by=["head_token", "COREF"], ascending=[True, True]).reset_index(drop=True)  # constructed from inside out
    for head_token, cat, COREF in non_zero_entities_df[['head_token', 'cat', 'COREF']].values:
        entity_tag = f"{entity_tag_prefix}{COREF}>"
        annotated_text_dict[head_token] = f"{annotated_text_dict[head_token]} {entity_tag}"

    annotated_tokens = list(annotated_text_dict.values())
    return annotated_tokens

In [ ]:
data_directory = "/data-lachesis/common/shared_tasks/crac26/crac26_data/"

annotated_plaintext_directory = "annotated_plaintext"
os.makedirs(annotated_plaintext_directory, exist_ok=True)

In [ ]:
annotation_type = "CRAC_provided_plaintext"
annotation_type = "XML_tags"
annotation_type = "simplified_XML_tags"
annotation_type = "head_only"

for split in ["llm-gold-train", "llm-gold-minidev", "llm-input_blind-minidev"]:
# for split in ["llm-input_blind-minitest"]:

    split_directory = Path(data_directory, split)
    ext = ".tokens_entities_dataset_json"  # change if needed
    datasets = sorted([p.stem for p in Path(split_directory).iterdir()
                       if p.is_file() and p.suffix == ext], reverse=False)

    split_annotated_plaintext = {}

    for DATASET_name in tqdm(datasets):
        DATASET_tokens_entities_dataset = json.load(open(f"{split_directory}/{DATASET_name}{ext}"))

        split_annotated_plaintext[DATASET_name] = []

        for DOC_id, DOC_tokens_entities_dict in enumerate(DATASET_tokens_entities_dataset):
            DOC_annotated_plaintext = []
            for sent_ID, SENT_tokens_entities_dict in enumerate(DOC_tokens_entities_dict):
                SENT_tokens, SENT_entities = SENT_tokens_entities_dict["tokens"], SENT_tokens_entities_dict["entities"]
                SENT_tokens = [token.replace(" ", "_") for token in SENT_tokens]

                if SENT_entities == []:
                    annotated_tokens = SENT_tokens
                else:
                    SENT_entities_df = pd.DataFrame(SENT_entities)
                    SENT_entities_df["mention_len"] = SENT_entities_df["end_token"] - SENT_entities_df["start_token"] + 1
                    if annotation_type == "XML_tags":
                        annotated_tokens = generate_annotated_tokens(SENT_tokens, SENT_entities_df,
                                                                     opening_ent="<ent id=COREF_", closing_ent="</ent>", opening_zero_ent="</zero_ent id=COREF_")
                    elif annotation_type == "simplified_XML_tags":
                        annotated_tokens = generate_annotated_tokens(SENT_tokens, SENT_entities_df,
                                                                     opening_ent="<ent", closing_ent="</ent>", opening_zero_ent="</zero")
                    elif annotation_type == "head_only":
                        annotated_tokens = generate_head_only_annotations(SENT_tokens, SENT_entities_df)

                    elif annotation_type == "CRAC_provided_plaintext":
                        annotated_tokens = CRAC_plaintext_annotation(SENT_tokens, SENT_entities_df)


                DOC_annotated_plaintext.append({"input": SENT_tokens,
                                                "gold": annotated_tokens})

            # print(annotated_tokens)
            split_annotated_plaintext[DATASET_name].append(DOC_annotated_plaintext)


    plaintext_file_path = os.path.join(annotated_plaintext_directory, f"{split}_{annotation_type}")
    with open(plaintext_file_path, "w", encoding="utf-8") as f:
        json.dump(split_annotated_plaintext, f, indent=4, ensure_ascii=False)